In [1]:
import sys
sys.path.append("../")

##### import library

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
import utility
import importlib
importlib.reload(utility)
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast,  get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

##### Read pickle

In [4]:
with open("HCDT_same_question_response_final.pkl", "rb") as f:
    HCDT_same_question_response = pickle.load(f)


In [5]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTarget", "BioChatter", "ctd", "ttd", "hcdt"]:

    
    # if model=="llama-3.3-70b-versatile":
    #     dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    # if model=="gemini-2.5-flash-lite":
    #     dct_result[model] = Gemini_same_question_response[model]

    # if model=="OpenTarget":
    #     dct_result[model] = Opentarget_same_question_response[model]

    # if model=="BioChatter":
    #     dct_result[model] = Biochatter_same_question_response[model]

    # if model=="ctd":
    #     dct_result[model] = CTD_same_question_response[model]

    if model=="hcdt":
        dct_result[model] = HCDT_same_question_response[model]


##### Verify if only valid columns are present

In [6]:
allowed = {"gene_name", "drug_name", "disease_name", "pathway_name"}

missing_df = []
not_dataframe = []
empty_df = []
bad_columns = []

total_payloads = 0

for model, queries in dct_result.items():
    if not isinstance(queries, dict):
        continue

    for question, runs in queries.items():
        if not isinstance(runs, dict):
            continue

        for run_id, payload in runs.items():
            total_payloads += 1

            if not isinstance(payload, dict) or "dataframe" not in payload:
                missing_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "payload_type": type(payload).__name__
                })
                continue

            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame):
                not_dataframe.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "df_type": type(df).__name__
                })
                continue

            if df.empty:
                empty_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns)
                })

            cols = {str(c) for c in df.columns}
            extra = sorted(cols - allowed)

            if extra:
                bad_columns.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns),
                    "extra_columns": extra
                })

print(f"Total payloads checked: {total_payloads}")
print(f"Missing dataframe key: {len(missing_df)}")
print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
print(f"Empty dataframes: {len(empty_df)}")
print(f"Dataframes with disallowed columns: {len(bad_columns)}")

bad_columns_df = pd.DataFrame(bad_columns)
missing_df_df = pd.DataFrame(missing_df)
not_dataframe_df = pd.DataFrame(not_dataframe)
empty_df_df = pd.DataFrame(empty_df)

display(bad_columns_df)
display(missing_df_df)
display(not_dataframe_df)
display(empty_df_df)


Total payloads checked: 295
Missing dataframe key: 0
Dataframe is not pd.DataFrame: 0
Empty dataframes: 0
Dataframes with disallowed columns: 0


""


""


""


""


In [7]:
dct_jaccard = dict()

question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name"}

for model, queries in dct_result.items():
    dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():

        for run_id, payload in runs.items():
            df = payload.get("dataframe")

            # ---- validation ----
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            # ---- skip pathway outputs ----
            if "pathway_name" in df.columns:
                continue

            # ---- column sanity ----
            if not set(df.columns).issubset(allowed_cols):
                continue

            # ---- collect entities globally ----
            for col in allowed_cols:
                if col in df.columns:
                    values = (
                        df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    question_entities[col].update(values)

# ---- finalize (sets → sorted lists) ----
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])


Working for model: hcdt


##### Associate id with all gene entry

In [8]:
# Resolve unique gene symbols to IDs (MyGene -> OpenTargets fallback).
df_gene = resolve_ensembl_ids_with_fallback(
    list(question_entities["gene_name"]),
    use_opentargets_fallback=True,
)
df_gene = df_gene.drop_duplicates(subset=["gene_name", "gene_id"]).reset_index(drop=True)
df_gene


2026-02-27 18:16:33,911 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:16:33,912 [WARNING] Input sequence provided is already in string format. No operation performed
2026-02-27 18:16:33,912 [INFO] querying 1-1000 ...
2026-02-27 18:16:36,375 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 18:16:37,382 [INFO] querying 1001-2000 ...
2026-02-27 18:16:38,722 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 18:16:39,728 [INFO] querying 2001-2987 ...
2026-02-27 18:16:41,049 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
2026-02-27 18:16:42,054 [INFO] Finished.
2026-02-27 18:16:42,197 [WARNING] 6 input query terms found dup hits:	[('CAST', 2), ('GLUD1P3', 2), ('GNRHR2', 2), ('ORAI1', 2), ('TEC', 2), ('ZNF542P', 2)]


,gene_name,gene_id,source
0,A1BG,ENSG00000121410,mygene
1,A2M,ENSG00000175899,mygene
2,AADAC,ENSG00000114771,mygene
3,AADAT,ENSG00000109576,mygene
4,AAK1,ENSG00000115977,mygene
...,...,...,...
2982,ZMPSTE24,ENSG00000084073,mygene
2983,ZNF423,ENSG00000102935,mygene
2984,ZNF542P,ENSG00000290720,mygene
2985,ZNF613,ENSG00000176024,mygene


In [9]:
df_gene[df_gene.isna().any(axis=1)]

,gene_name,gene_id,source


##### Associate id with all drug entry

In [10]:
# Resolve unique drug surface forms to IDs.
# This keeps original raw names as rows and writes mapped ID + source.
df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
df_drug = df_drug.drop_duplicates(subset=["drug_name", "drug_id"]).reset_index(drop=True)
df_drug


2026-02-27 18:19:02,222 [INFO] [LLM] Total 10296 names split into 167 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:19:02,226 [INFO] [LLM] Batch size: 43
2026-02-27 18:19:02,259 [INFO] [LLM] Batch size: 60
2026-02-27 18:19:02,264 [INFO] [LLM] Batch size: 56
2026-02-27 18:19:02,267 [INFO] [LLM] Batch size: 46
2026-02-27 18:19:02,268 [INFO] [LLM] Batch size: 50
2026-02-27 18:19:02,268 [INFO] [LLM] Batch size: 56
2026-02-27 18:19:02,269 [INFO] [LLM] Batch size: 52
2026-02-27 18:19:02,269 [INFO] [LLM] Batch size: 40
2026-02-27 18:19:02,270 [INFO] [LLM] Batch size: 60
2026-02-27 18:19:02,270 [INFO] [LLM] Batch size: 73
2026-02-27 18:19:02,271 [INFO] [LLM] Batch size: 53
2026-02-27 18:19:02,271 [INFO] [LLM] Batch size: 46
2026-02-27 18:19:02,271 [INFO] [LLM] Batch size: 60
2026-02-27 18:19:02,272 [INFO] [LLM] Batch size: 50
2026-02-27 18:19:02,272 [INFO] [LLM] Batch size: 53
2026-02-27 18:19:02,273 [INFO] [LLM] Batch size: 55
2026-02-27 18:1

[map][drug][OpenTargets] raw='A-623' -> id='CHEMBL2107877'
[map][drug][OpenTargets] raw='AC-1204' -> id='CHEMBL1406148'
[map][drug][OpenTargets] raw='ADL-5859' -> id='CHEMBL494480'
[map][drug][OpenTargets] raw='AMANTADINE' -> id='CHEMBL660'
[map][drug][OpenTargets] raw='AMINOSALICYLIC ACID' -> id='CHEMBL1169'
[map][drug][OpenTargets] raw='AMPICILLIN' -> id='CHEMBL174'
[map][drug][OpenTargets] raw='AV-412' -> id='CHEMBL6067986'
[map][drug][OpenTargets] raw='AV-412 free base' -> id='CHEMBL2138625'
[map][drug][OpenTargets] raw='Abemaciclib' -> id='CHEMBL3301610'
[map][drug][OpenTargets] raw='Abivertinib' -> id='CHEMBL4297865'
[map][drug][OpenTargets] raw='Acalabrutinib' -> id='CHEMBL3707348'
[map][drug][OpenTargets] raw='Acetaminophen' -> id='CHEMBL112'
[map][drug][OpenTargets] raw='Afatinib' -> id='CHEMBL1173655'
[map][drug][OpenTargets] raw='Afatinib Dimaleate' -> id='CHEMBL2105712'
[map][drug][OpenTargets] raw='Agerafenib' -> id='CHEMBL2029988'
[map][drug][OpenTargets] raw='Ajulemic ac

2026-02-27 18:19:07,564 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:19:07,588 [INFO] [LLM] Batch done in 5.33s
2026-02-27 18:19:07,614 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:19:07,616 [INFO] [LLM] Batch done in 5.35s
2026-02-27 18:19:07,847 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:19:07,850 [INFO] [LLM] Batch done in 5.62s
2026-02-27 18:19:13,111 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:19:13,114 [INFO] [LLM] Batch done in 10.85s
2026-02-27 18:19:15,371 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:19:15,373 [INFO] [LLM] Batch done in 13.10s
2026-02-27 18:19:18,149 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:19:

[map][drug][Llama] raw='(S)-Ibrutinib' -> canonical='IBRUTINIB' -> id='CHEMBL1873475'
[map][drug][Llama] raw='Vandetanib hydrochloride' -> canonical='VANDETANIB' -> id='CHEMBL24828'
[map][drug][Llama] raw='diammine[cyclobutane-1,1-dicarboxylato(2-)-k2O1,O1]platinum' -> canonical='CARBOPLATIN' -> id='CHEMBL1351'


,drug_name,drug_id,source
0,((plusmn))-Zanubrutinib,None,None
1,"(+)-(1R,3R)-1-(3-hydroxy-cyclopentyl)-3-(4-met...",None,None
2,(+)-Interiotherin B,None,None
3,"(+/-)-trans-3,4,4a,5,6,10b-Hexahydro-9-carbamo...",None,None
4,"(-)-(1S,3S)-1-(3-hydroxy-cyclopentyl)-3-(4-met...",None,None
...,...,...,...
10865,"{N}-[4-[({E})-3-(3,4,5-trimethoxyphenyl)prop-2...",None,None
10866,"{N}-[4-[({E})-3-(3,4-dimethoxyphenyl)prop-2-en...",None,None
10867,"{N}-[4-[({E})-3-(3,4-dimethoxyphenyl)prop-2-en...",None,None
10868,"{N}-[4-[({E})-3-(3,4-dimethoxyphenyl)prop-2-en...",None,None


In [11]:
df_drug["source"].value_counts()

source
OpenTargets    574
Llama            3
Name: count, dtype: int64

In [12]:
# df_drug[df_drug["source"]=="Llama"]

In [13]:
df_drug[df_drug.isna().any(axis=1)]

,drug_name,drug_id,source
0,((plusmn))-Zanubrutinib,None,None
1,"(+)-(1R,3R)-1-(3-hydroxy-cyclopentyl)-3-(4-met...",None,None
2,(+)-Interiotherin B,None,None
3,"(+/-)-trans-3,4,4a,5,6,10b-Hexahydro-9-carbamo...",None,None
4,"(-)-(1S,3S)-1-(3-hydroxy-cyclopentyl)-3-(4-met...",None,None
...,...,...,...
10865,"{N}-[4-[({E})-3-(3,4,5-trimethoxyphenyl)prop-2...",None,None
10866,"{N}-[4-[({E})-3-(3,4-dimethoxyphenyl)prop-2-en...",None,None
10867,"{N}-[4-[({E})-3-(3,4-dimethoxyphenyl)prop-2-en...",None,None
10868,"{N}-[4-[({E})-3-(3,4-dimethoxyphenyl)prop-2-en...",None,None


##### Associate id with all disease entry

In [14]:
# Resolve unique disease surface forms to IDs.
# Canonical fallback is internal; returned rows remain keyed by raw input disease_name.
df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
df_disease = df_disease.drop_duplicates(subset=["disease_name", "disease_id"]).reset_index(drop=True)
df_disease


2026-02-27 18:24:13,427 [INFO] [LLM] Total 95 names split into 2 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:24:13,428 [INFO] [LLM] Batch size: 80
2026-02-27 18:24:13,429 [INFO] [LLM] Batch size: 15


[map][disease][OpenTargets] raw='5q- syndrome' -> id='MONDO_0007925'
[map][disease][OpenTargets] raw='Absence seizure' -> id='MONDO_0010826'
[map][disease][OpenTargets] raw='Acidosis' -> id='EFO_1000014'
[map][disease][OpenTargets] raw='Acne vulgaris' -> id='EFO_0003894'
[map][disease][OpenTargets] raw='Acquired methemoglobinemia' -> id='MONDO_0018740'
[map][disease][OpenTargets] raw='Acquired pure red cell aplasia' -> id='MONDO_0020338'
[map][disease][OpenTargets] raw='Acromegaly' -> id='EFO_1001485'
[map][disease][OpenTargets] raw='Acute Coronary Syndrome' -> id='EFO_0005672'
[map][disease][OpenTargets] raw='Acute Kidney Injury' -> id='MONDO_0002492'
[map][disease][OpenTargets] raw='Acute Lung Injury' -> id='EFO_1000637'
[map][disease][OpenTargets] raw='Acute myelogenous leukaemia' -> id='EFO_0000222'
[map][disease][OpenTargets] raw='Acute myeloid leukaemia' -> id='MONDO_0015667'
[map][disease][OpenTargets] raw='Acute myeloid leukemia' -> id='EFO_0000222'
[map][disease][OpenTargets] 

2026-02-27 18:24:14,273 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:24:14,275 [INFO] [LLM] Batch done in 0.84s
2026-02-27 18:24:16,964 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:24:16,966 [INFO] [LLM] Batch done in 3.54s
2026-02-27 18:24:16,967 [INFO] [LLM] 'Acid-reflux disorder' → 'gastroesophageal reflux disease'
2026-02-27 18:24:16,968 [INFO] [LLM] 'Acute heart failure' → 'heart failure'
2026-02-27 18:24:16,968 [INFO] [LLM] 'Acute lymphoblastic leukaemia' → 'acute lymphoblastic leukemia'
2026-02-27 18:24:16,969 [INFO] [LLM] 'Acute pain' → None
2026-02-27 18:24:16,970 [INFO] [LLM] 'Allograft rejection' → 'allograft rejection'
2026-02-27 18:24:16,970 [INFO] [LLM] 'Amaurosis Fugax' → 'amaurosis fugax'
2026-02-27 18:24:16,971 [INFO] [LLM] 'Amyotrophic lateral sclerosis (ALS)' → 'amyotrophic lateral sclerosis'
2026-02-27 18:24:16,971 [INFO] [LLM] 'Anabolic metaboli

[map][disease][Llama] raw='Acid-reflux disorder' -> canonical='gastroesophageal reflux disease' -> id='EFO_0003948'
[map][disease][Llama] raw='Acute heart failure' -> canonical='heart failure' -> id='EFO_0003144'
[map][disease][Llama] raw='Acute lymphoblastic leukaemia' -> canonical='acute lymphoblastic leukemia' -> id='EFO_0000220'
[map][disease][Llama] raw='Amyotrophic lateral sclerosis (ALS)' -> canonical='amyotrophic lateral sclerosis' -> id='MONDO_0004976'
[map][disease][Llama] raw='Anaplastic large-cell lymphoma' -> canonical='anaplastic large cell lymphoma' -> id='EFO_0003032'
[map][disease][Llama] raw='Attention deficit hyperactivity disorder (ADHD)' -> canonical='attention deficit hyperactivity disorder' -> id='EFO_0003888'
[map][disease][Llama] raw='Congestive cardiac insufficiency' -> canonical='heart failure' -> id='EFO_0003144'
[map][disease][Llama] raw='Erythropoietic porphyrias' -> canonical='porphyria' -> id='MONDO_0037939'
[map][disease][Llama] raw='Female hypogonadism

,disease_name,disease_id,source
0,5q- syndrome,MONDO_0007925,OpenTargets
1,Absence seizure,MONDO_0010826,OpenTargets
2,Acid-reflux disorder,EFO_0003948,Llama
3,Acidosis,EFO_1000014,OpenTargets
4,Acne vulgaris,EFO_0003894,OpenTargets
...,...,...,...
556,West syndrome,MONDO_0018097,OpenTargets
557,Wilson disease,MONDO_0010200,OpenTargets
558,Worm infection,EFO_1001342,OpenTargets
559,Xerostomia,EFO_0009869,OpenTargets


In [15]:
df_disease["source"].value_counts()

source
OpenTargets    466
Llama           46
Name: count, dtype: int64

In [16]:
df_disease[df_disease.isna().any(axis=1)]

,disease_name,disease_id,source
16,Acute pain,None,None
25,Allograft rejection,None,None
28,Amaurosis Fugax,None,None
34,Anabolic metabolism,None,None
35,Anaesthesia,None,None
43,Anomia,None,None
78,Bone metastases,None,None
100,Cestodes infection,None,None
105,Chronic low back pain,None,None
140,Death,None,None


In [17]:
# ------------------------------------------------------------------
# Disease canonical audit (OpenTargets-strict IDs only)
# ------------------------------------------------------------------
# IMPORTANT:
# - This cell runs an additional Llama canonicalization pass over unresolved rows.
# - IDs are still resolved ONLY via OpenTargets mapIds (score==1).
# - To avoid changing the baseline resolver output, writeback is OFF by default.
APPLY_AUDIT_MAPPINGS = False

# 1) Rows unresolved after primary resolver
unresolved_rows = df_disease[df_disease["disease_id"].isna()].reset_index(drop=True)
unresolved_raw_names = list(
    dict.fromkeys(
        str(name).strip()
        for name in unresolved_rows["disease_name"].dropna().astype(str).tolist()
        if str(name).strip()
    )
)

# 2) Llama canonicalization suggestions
disease_input_map = {name: None for name in unresolved_raw_names}
canonicalisation_map = await utility.canonicalise_disease_dict(disease_input_map)

# 3) OpenTargets strict verification (score==1 only)
canonical_terms_norm = list(
    dict.fromkeys(
        utility._normalize_disease_term(v)
        for v in canonicalisation_map.values()
        if isinstance(v, str) and v.strip()
    )
)
canonical_to_id = utility.resolve_diseases_opentargets_bulk(
    canonical_terms_norm,
    strict_score_one=True,
)

# 4) Build persistent audit table
audit_rows = []
applied_count = 0
candidate_count = 0

for raw_disease_name in unresolved_raw_names:
    canonical_name = canonicalisation_map.get(raw_disease_name)
    if isinstance(canonical_name, str) and canonical_name.strip():
        canonical_name = canonical_name.strip()
        canonical_norm = utility._normalize_disease_term(canonical_name)
        resolved_id = canonical_to_id.get(canonical_norm)
    else:
        canonical_name = None
        canonical_norm = None
        resolved_id = None

    can_apply = bool(resolved_id)
    if can_apply:
        candidate_count += 1
        print(f"[audit][disease][Llama->OT] raw='{raw_disease_name}' -> canonical='{canonical_name}' -> id='{resolved_id}'")

    if can_apply and APPLY_AUDIT_MAPPINGS:
        mask = df_disease["disease_name"].astype(str).str.strip() == raw_disease_name
        df_disease.loc[mask, "disease_id"] = resolved_id
        df_disease.loc[mask, "source"] = "Llama"
        applied_count += 1

    status = (
        "mappable_ot_exact" if can_apply else
        ("no_canonical" if canonical_name is None else "canonical_not_in_ot")
    )
    audit_rows.append({
        "raw_disease_name": raw_disease_name,
        "canonical_name": canonical_name,
        "canonical_norm": canonical_norm,
        "resolved_disease_id": resolved_id,
        "status": status,
        "can_apply": can_apply,
        "applied": bool(can_apply and APPLY_AUDIT_MAPPINGS),
    })

disease_canonical_audit_df = pd.DataFrame(audit_rows)
if not disease_canonical_audit_df.empty:
    disease_canonical_audit_df = disease_canonical_audit_df.sort_values(
        ["status", "raw_disease_name"],
        ascending=[True, True],
    ).reset_index(drop=True)

print(f"[summary][disease][audit] OT-mappable candidates: {candidate_count}")
print(f"[summary][disease][audit] applied to df_disease: {applied_count} (APPLY_AUDIT_MAPPINGS={APPLY_AUDIT_MAPPINGS})")
if not disease_canonical_audit_df.empty:
    print("[summary][disease][audit] status counts:")
    print(disease_canonical_audit_df["status"].value_counts(dropna=False))

disease_canonical_audit_df


2026-02-27 18:24:18,185 [INFO] [LLM] Total 49 names split into 1 batches (max_batch_chars=6000, max_batch_items=80, model=llama-3.3-70b-versatile).
2026-02-27 18:24:18,186 [INFO] [LLM] Batch size: 49
2026-02-27 18:24:19,350 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-02-27 18:24:19,352 [INFO] [LLM] Batch done in 1.17s
2026-02-27 18:24:19,353 [INFO] [LLM] 'Acute pain' → None
2026-02-27 18:24:19,354 [INFO] [LLM] 'Allograft rejection' → 'allograft rejection'
2026-02-27 18:24:19,354 [INFO] [LLM] 'Amaurosis Fugax' → 'amaurosis fugax'
2026-02-27 18:24:19,355 [INFO] [LLM] 'Anabolic metabolism' → None
2026-02-27 18:24:19,355 [INFO] [LLM] 'Anaesthesia' → None
2026-02-27 18:24:19,356 [INFO] [LLM] 'Anomia' → None
2026-02-27 18:24:19,356 [INFO] [LLM] 'Bone metastases' → None
2026-02-27 18:24:19,357 [INFO] [LLM] 'Cestodes infection' → None
2026-02-27 18:24:19,357 [INFO] [LLM] 'Chronic low back pain' → None
2026-02-27 18:24:19,358 [INFO] [LLM] 'De

[audit][disease][Llama->OT] raw='Prion Diseases' -> canonical='prion disease' -> id='EFO_0004720'
[audit][disease][Llama->OT] raw='Squamous head and neck cell carcinom' -> canonical='head and neck squamous cell carcinoma' -> id='EFO_0000181'
[audit][disease][Llama->OT] raw='Status epilepticus seizure' -> canonical='status epilepticus' -> id='EFO_0008526'
[audit][disease][Llama->OT] raw='Vertigo meniere disease' -> canonical='Meniere disease' -> id='EFO_0006862'
[summary][disease][audit] OT-mappable candidates: 4
[summary][disease][audit] applied to df_disease: 0 (APPLY_AUDIT_MAPPINGS=False)
[summary][disease][audit] status counts:
status
no_canonical         45
mappable_ot_exact     4
Name: count, dtype: int64


,raw_disease_name,canonical_name,canonical_norm,resolved_disease_id,status,can_apply,applied
0,Prion Diseases,prion disease,prion disease,EFO_0004720,mappable_ot_exact,True,False
1,Squamous head and neck cell carcinom,head and neck squamous cell carcinoma,head and neck squamous cell carcinoma,EFO_0000181,mappable_ot_exact,True,False
2,Status epilepticus seizure,status epilepticus,status epilepticus,EFO_0008526,mappable_ot_exact,True,False
3,Vertigo meniere disease,Meniere disease,meniere disease,EFO_0006862,mappable_ot_exact,True,False
4,Acute pain,None,None,None,no_canonical,False,False
5,Allograft rejection,None,None,None,no_canonical,False,False
6,Amaurosis Fugax,None,None,None,no_canonical,False,False
7,Anabolic metabolism,None,None,None,no_canonical,False,False
8,Anaesthesia,None,None,None,no_canonical,False,False
9,Anomia,None,None,None,no_canonical,False,False


In [18]:
# disease_canonical_audit_df

In [19]:
df_disease[df_disease["source"]=='Llama']

,disease_name,disease_id,source
2,Acid-reflux disorder,EFO_0003948,Llama
11,Acute heart failure,EFO_0003144,Llama
12,Acute lymphoblastic leukaemia,EFO_0000220,Llama
33,Amyotrophic lateral sclerosis (ALS),MONDO_0004976,Llama
37,Anaplastic large-cell lymphoma,EFO_0003032,Llama
62,Attention deficit hyperactivity disorder (ADHD),EFO_0003888,Llama
124,Congestive cardiac insufficiency,EFO_0003144,Llama
190,Erythropoietic porphyrias,MONDO_0037939,Llama
196,Female hypogonadism,MONDO_0002146,Llama
210,Gastrointestinal stromal tumour,EFO_0010279,Llama


In [20]:
df_disease["source"].value_counts()

source
OpenTargets    466
Llama           46
Name: count, dtype: int64

In [21]:
df_disease

,disease_name,disease_id,source
0,5q- syndrome,MONDO_0007925,OpenTargets
1,Absence seizure,MONDO_0010826,OpenTargets
2,Acid-reflux disorder,EFO_0003948,Llama
3,Acidosis,EFO_1000014,OpenTargets
4,Acne vulgaris,EFO_0003894,OpenTargets
...,...,...,...
556,West syndrome,MONDO_0018097,OpenTargets
557,Wilson disease,MONDO_0010200,OpenTargets
558,Worm infection,EFO_1001342,OpenTargets
559,Xerostomia,EFO_0009869,OpenTargets


In [22]:
df_disease[df_disease["source"]=="Llama"]

,disease_name,disease_id,source
2,Acid-reflux disorder,EFO_0003948,Llama
11,Acute heart failure,EFO_0003144,Llama
12,Acute lymphoblastic leukaemia,EFO_0000220,Llama
33,Amyotrophic lateral sclerosis (ALS),MONDO_0004976,Llama
37,Anaplastic large-cell lymphoma,EFO_0003032,Llama
62,Attention deficit hyperactivity disorder (ADHD),EFO_0003888,Llama
124,Congestive cardiac insufficiency,EFO_0003144,Llama
190,Erythropoietic porphyrias,MONDO_0037939,Llama
196,Female hypogonadism,MONDO_0002146,Llama
210,Gastrointestinal stromal tumour,EFO_0010279,Llama


In [23]:
def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # ------------------------------------------------------------------
    # Build deterministic one-to-one mapping tables
    # ------------------------------------------------------------------
    gene_map = (
        df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
        [["_gene_norm", "gene_id"]]
        .dropna(subset=["gene_id"])
        .drop_duplicates("_gene_norm", keep="first")
    )

    disease_map = (
        df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
        [["_disease_norm", "disease_id"]]
        .dropna(subset=["disease_id"])
        .drop_duplicates("_disease_norm", keep="first")
    )

    drug_map = (
        df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
        [["_drug_norm", "drug_id"]]
        .dropna(subset=["drug_id"])
        .drop_duplicates("_drug_norm", keep="first")
    )

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

                work_df = df.copy()

                # print(work_df.head())

                final_cols = []

                # ---- gene ----
                if "gene_name" in work_df.columns:
                    work_df["_gene_norm"] = (
                        work_df["gene_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
                    final_cols.append("gene_id")

                # ---- disease ----
                if "disease_name" in work_df.columns:
                    work_df["_disease_norm"] = (
                        work_df["disease_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
                    final_cols.append("disease_id")

                # ---- drug ----
                if "drug_name" in work_df.columns:
                    work_df["_drug_norm"] = (
                        work_df["drug_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
                    final_cols.append("drug_id")

                if not final_cols:
                    continue

                # ---- canonical ID dataframe ----
                id_df = (
                    work_df[final_cols]
                    .dropna(how="any")
                    .drop_duplicates()
                    .reset_index(drop=True)
                )

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)


Working for model: hcdt
'Which diseases are associated with BRAF?': J=1.0000, |∩|=145, |∪|=145, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=1.0000, |∩|=797, |∪|=797, runs=5
'Which diseases are associated with CDK4?': J=1.0000, |∩|=140, |∪|=140, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=102, |∪|=102, runs=5
'Which genes are associated with ovarian cancer?': J=1.0000, |∩|=803, |∪|=803, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=69, |∪|=69, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
Not enough valid runs for 'Which genes are associated with fever (pyrexia)?' (0/0). Skipping.
Not enough valid runs for 'Which diseases are treated with bevacizumab?' (0/0). Skipping.
'What are the known targets of cabozantinib?': J=1.0000, |∩|=89, |∪|=89, runs=5
Not en

In [24]:
# Run Jaccard consistency computation across runs after ID grounding.
dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)

# Flatten into a quick summary table for inspection.
rows = [
    {
        "model": model,
        "question": question,
        "jaccard": payload["jaccard"],
        "n_valid_runs": payload["n_valid_runs"],
        "intersection_size": len(payload["intersection"]),
        "union_size": len(payload["union"]),
    }
    for model, qmap in dct_jaccard.items()
    for question, payload in qmap.items()
]

jaccard_summary = pd.DataFrame(rows)
if jaccard_summary.empty:
    print("No valid Jaccard entries were produced. Check input runs and validation filters.")
else:
    display(jaccard_summary.sort_values(["model", "jaccard"], ascending=[True, False]).head(20))



Working for model: hcdt
'Which diseases are associated with BRAF?': J=1.0000, |∩|=145, |∪|=145, runs=5
'Which genes are associated with amyotrophic lateral sclerosis?': J=1.0000, |∩|=797, |∪|=797, runs=5
'Which diseases are associated with CDK4?': J=1.0000, |∩|=140, |∪|=140, runs=5
Not enough valid runs for 'Which pathways are associated with the JAK2 gene?' (0/5). Skipping.
'Which drugs are used to treat rheumatoid arthritis?': J=1.0000, |∩|=102, |∪|=102, runs=5
'Which genes are associated with ovarian cancer?': J=1.0000, |∩|=803, |∪|=803, runs=5
'What are the known targets of regorafenib?': J=1.0000, |∩|=69, |∪|=69, runs=5
Not enough valid runs for 'Which pathways are associated with the STAT3 gene?' (0/5). Skipping.
Not enough valid runs for 'Which genes are associated with fever (pyrexia)?' (0/0). Skipping.
Not enough valid runs for 'Which diseases are treated with bevacizumab?' (0/0). Skipping.
'What are the known targets of cabozantinib?': J=1.0000, |∩|=89, |∪|=89, runs=5
Not en

,model,question,jaccard,n_valid_runs,intersection_size,union_size
0,hcdt,Which diseases are associated with BRAF?,1.0,5,145,145
1,hcdt,Which genes are associated with amyotrophic la...,1.0,5,797,797
2,hcdt,Which diseases are associated with CDK4?,1.0,5,140,140
3,hcdt,Which drugs are used to treat rheumatoid arthr...,1.0,5,102,102
4,hcdt,Which genes are associated with ovarian cancer?,1.0,5,803,803
5,hcdt,What are the known targets of regorafenib?,1.0,5,69,69
6,hcdt,What are the known targets of cabozantinib?,1.0,5,89,89
7,hcdt,Which diseases are associated with the NPM1 gene?,1.0,5,14,14
8,hcdt,Which drugs are used to treat migraine?,1.0,5,34,34
9,hcdt,What drugs are used to treat rickets?,1.0,5,2,2


In [25]:
# Persist enriched run payloads with dataframe_id attached for each run.
with open("HCDT_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)
